In [1]:
import pandas as pd
import numpy as np

# --- Load the filtered catalogue subset ---
df = pd.read_csv("filtered_events.csv")

# --- Classify each event into its observing run (O3 vs O4) from its name ---
# GW19* and GW20* (up to O3b, ending March 2020) -> O3
# GW23*, GW24*, GW25*                             -> O4
def classify_run(name: str) -> str:
    yy = name.replace("GW", "")[:2]
    if yy in ("19", "20"):
        return "O3"
    elif yy in ("23", "24", "25"):
        return "O4"
    else:
        return "unknown"

df["run"] = df["name"].apply(classify_run)

# --- Detector-frame chirp mass ---
# GWOSC convention: 'chirp_mass' (when populated) is already detector-frame.
# Where it's missing, derive it from the source-frame chirp mass and redshift:
#   M_det = M_source * (1 + z)
df["chirp_mass_det"] = df["chirp_mass"]
needs_derivation = df["chirp_mass_det"].isna()
df.loc[needs_derivation, "chirp_mass_det"] = (
    df.loc[needs_derivation, "chirp_mass_source"]
    * (1 + df.loc[needs_derivation, "redshift"])
)

# Flag any events for which neither route worked (e.g. missing redshift)
missing = df[df["chirp_mass_det"].isna()]
if not missing.empty:
    print(f"WARNING: could not compute detector-frame chirp mass for "
          f"{len(missing)} event(s): {missing['name'].tolist()}")

# --- Split into O3 / O4 dataframes ---
cols = ["name", "run", "chirp_mass_source", "redshift", "chirp_mass_det"]
o3_df = df[df["run"] == "O3"][cols].reset_index(drop=True)
o4_df = df[df["run"] == "O4"][cols].reset_index(drop=True)

print(f"O3 events: {len(o3_df)}")
display(o3_df)

print(f"\nO4 events: {len(o4_df)}")
display(o4_df)

O3 events: 24


,name,run,chirp_mass_source,redshift,chirp_mass_det
0,GW190408_181802,O3,18.0,0.30,24.000
1,GW190412,O3,13.3,0.15,15.200
2,GW190413_134308,O3,34.0,0.60,60.000
3,GW190421_213856,O3,31.0,0.50,47.000
4,GW190503_185404,O3,31.0,0.31,41.000
5,GW190512_180714,O3,14.4,0.29,19.000
6,GW190513_205428,O3,23.0,0.42,32.000
7,GW190519_153544,O3,43.0,0.50,70.000
8,GW190521,O3,63.3,0.56,98.748
9,GW190521_074359,O3,33.0,0.21,40.000



O4 events: 72


,name,run,chirp_mass_source,redshift,chirp_mass_det
0,GW230601_224134,O4,45.10,0.60,72.1600
1,GW230606_004305,O4,26.50,0.48,39.2200
2,GW230608_205047,O4,32.70,0.58,51.6660
3,GW230609_064958,O4,25.60,0.56,39.9360
4,GW230627_015337,O4,6.02,0.07,6.4414
...,...,...,...,...,...
67,GW250109_010541,O4,27.40,0.56,42.7440
68,GW250114_082203,O4,28.71,0.09,31.2939
69,GW250118_170523,O4,29.00,0.35,39.1500
70,GW250119_025138,O4,25.70,0.52,39.0640


In [2]:
# --- Min/max detector-frame chirp mass per run ---
for run_name, run_df in [("O3", o3_df), ("O4", o4_df)]:
    valid = run_df.dropna(subset=["chirp_mass_det"])
    max_row = valid.loc[valid["chirp_mass_det"].idxmax()]
    min_row = valid.loc[valid["chirp_mass_det"].idxmin()]
    print(f"\n{run_name} detector-frame chirp mass range:")
    print(f"  Max: {max_row['chirp_mass_det']:.2f} Msun  ({max_row['name']})")
    print(f"  Min: {min_row['chirp_mass_det']:.2f} Msun  ({min_row['name']})")

# --- Min/max luminosity distance per run ---
for run_name, run_df in [("O3", o3_df), ("O4", o4_df)]:
    valid = run_df.dropna(subset=["redshift"])
    max_row = valid.loc[valid["redshift"].idxmax()]
    min_row = valid.loc[valid["redshift"].idxmin()]
    #convert redshift to luminosity distance using astropy.cosmology
    from astropy.cosmology import Planck18 as cosmo
    max_row["luminosity_distance"] = cosmo.luminosity_distance(max_row["redshift"]).value
    min_row["luminosity_distance"] = cosmo.luminosity_distance(min_row["redshift"]).value
    print(f"\n{run_name} redshift range:")
    print(f"  Max: {max_row['redshift']:.3f}  ({max_row['name']})")
    print(f"  Min: {min_row['redshift']:.3f}  ({min_row['name']})")
    print(f"\n{run_name} luminosity distance range:")
    print(f"  Max: {max_row['luminosity_distance']:.2f} Mpc  ({max_row['name']})")
    print(f"  Min: {min_row['luminosity_distance']:.2f} Mpc  ({min_row['name']})")


O3 detector-frame chirp mass range:
  Max: 98.75 Msun  (GW190521)
  Min: 9.80 Msun  (GW190707_093326)

O4 detector-frame chirp mass range:
  Max: 141.40 Msun  (GW231123_135430)
  Min: 6.44 Msun  (GW230627_015337)

O3 redshift range:
  Max: 0.800  (GW190706_222641)
  Min: 0.150  (GW190412)

O3 luminosity distance range:
  Max: 5162.17 Mpc  (GW190706_222641)
  Min: 736.92 Mpc  (GW190412)

O4 redshift range:
  Max: 0.960  (GW230922_040658)
  Min: 0.070  (GW230627_015337)

O4 luminosity distance range:
  Max: 6458.29 Mpc  (GW230922_040658)
  Min: 326.38 Mpc  (GW230627_015337)


/tmp/ipykernel_778104/668297910.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  max_row["luminosity_distance"] = cosmo.luminosity_distance(max_row["redshift"]).value
/tmp/ipykernel_778104/668297910.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  max_row["luminosity_distance"] = cosmo.luminosity_distance(max_row["redshift"]).value
/tmp/ipykernel_778104/668297910.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  min_row["lum

In [3]:
# --- Detector-frame individual component masses ---
# The catalogue only provides source-frame mass_1/mass_2, so derive
# detector-frame masses the same way: m_det = m_source * (1 + z)
df["mass_1_det"] = df["mass_1_source"] * (1 + df["redshift"])
df["mass_2_det"] = df["mass_2_source"] * (1 + df["redshift"])

# Refresh O3/O4 split to include the new columns
cols_indiv = ["name", "run", "mass_1_source", "mass_2_source", "redshift",
              "mass_1_det", "mass_2_det"]
o3_df_indiv = df[df["run"] == "O3"][cols_indiv].reset_index(drop=True)
o4_df_indiv = df[df["run"] == "O4"][cols_indiv].reset_index(drop=True)

print(f"O3 events: {len(o3_df_indiv)}")
display(o3_df_indiv)

print(f"\nO4 events: {len(o4_df_indiv)}")
display(o4_df_indiv)

# --- Min/max detector-frame individual mass per run ---
# Combine mass_1_det and mass_2_det into one pool per run, since "mass range"
# spans both companions, not just the primary.
for run_name, run_df in [("O3", o3_df_indiv), ("O4", o4_df_indiv)]:
    pooled = pd.concat([
        run_df[["name", "mass_1_det"]].rename(columns={"mass_1_det": "mass_det"}),
        run_df[["name", "mass_2_det"]].rename(columns={"mass_2_det": "mass_det"}),
    ]).dropna(subset=["mass_det"]).reset_index(drop=True)

    max_row = pooled.loc[pooled["mass_det"].idxmax()]
    min_row = pooled.loc[pooled["mass_det"].idxmin()]
    print(f"\n{run_name} detector-frame individual mass range (across mass_1 & mass_2):")
    print(f"  Max: {max_row['mass_det']:.2f} Msun  ({max_row['name']})")
    print(f"  Min: {min_row['mass_det']:.2f} Msun  ({min_row['name']})")

O3 events: 24


,name,run,mass_1_source,mass_2_source,redshift,mass_1_det,mass_2_det
0,GW190408_181802,O3,25.0,18.0,0.30,32.500,23.400
1,GW190412,O3,31.0,8.0,0.15,35.650,9.200
2,GW190413_134308,O3,56.0,29.0,0.60,89.600,46.400
3,GW190421_213856,O3,43.0,31.0,0.50,64.500,46.500
4,GW190503_185404,O3,42.0,31.0,0.31,55.020,40.610
5,GW190512_180714,O3,24.0,12.0,0.29,30.960,15.480
6,GW190513_205428,O3,41.0,18.0,0.42,58.220,25.560
7,GW190519_153544,O3,62.0,41.0,0.50,93.000,61.500
8,GW190521,O3,98.4,57.2,0.56,153.504,89.232
9,GW190521_074359,O3,43.0,35.0,0.21,52.030,42.350



O4 events: 72


,name,run,mass_1_source,mass_2_source,redshift,mass_1_det,mass_2_det
0,GW230601_224134,O4,64.00,44.00,0.60,102.4000,70.4000
1,GW230606_004305,O4,37.60,25.70,0.48,55.6480,38.0360
2,GW230608_205047,O4,48.00,31.00,0.58,75.8400,48.9800
3,GW230609_064958,O4,35.40,25.20,0.56,55.2240,39.3120
4,GW230627_015337,O4,8.40,5.74,0.07,8.9880,6.1418
...,...,...,...,...,...,...,...
67,GW250109_010541,O4,37.50,27.40,0.56,58.5000,42.7440
68,GW250114_082203,O4,33.76,32.26,0.09,36.7984,35.1634
69,GW250118_170523,O4,38.10,29.60,0.35,51.4350,39.9600
70,GW250119_025138,O4,33.80,26.70,0.52,51.3760,40.5840



O3 detector-frame individual mass range (across mass_1 & mass_2):
  Max: 153.50 Msun  (GW190521)
  Min: 9.20 Msun  (GW190412)

O4 detector-frame individual mass range (across mass_1 & mass_2):
  Max: 191.80 Msun  (GW231123_135430)
  Min: 6.14 Msun  (GW230627_015337)


In [7]:

# --- Detector-frame individual component masses ---
# The catalogue only provides source-frame mass_1/mass_2, so derive
# detector-frame masses the same way: m_det = m_source * (1 + z)
df["mass_1_det"] = df["mass_1_source"] * (1 + df["redshift"])
df["mass_2_det"] = df["mass_2_source"] * (1 + df["redshift"])

# Refresh O3/O4 split to include the new columns
cols_indiv = ["name", "run", "mass_1_source", "mass_2_source", "redshift",
              "mass_1_det", "mass_2_det"]
o3_df_indiv = df[df["run"] == "O3"][cols_indiv].reset_index(drop=True)
o4_df_indiv = df[df["run"] == "O4"][cols_indiv].reset_index(drop=True)

# print the events with chirp mass below 20
for run_name, run_df in [("O3", o3_df), ("O4", o4_df)]:
    low_chirp_mass = run_df[run_df["chirp_mass_det"] < 20]
    print(f"\n{run_name} events with detector-frame chirp mass < 20 Msun:")
    display(low_chirp_mass)

# print the events with chirp mass above 100
for run_name, run_df in [("O3", o3_df), ("O4", o4_df)]:
    high_chirp_mass = run_df[run_df["chirp_mass_det"] > 100]
    print(f"\n{run_name} events with detector-frame chirp mass > 100 Msun:")
    display(high_chirp_mass)


O3 events with detector-frame chirp mass < 20 Msun:


,name,run,chirp_mass_source,redshift,chirp_mass_det
1,GW190412,O3,13.3,0.15,15.200
5,GW190512_180714,O3,14.4,0.29,19.000
12,GW190707_093326,O3,8.5,0.17,9.800
22,GW200225_060421,O3,14.2,0.22,17.324



O4 events with detector-frame chirp mass < 20 Msun:


,name,run,chirp_mass_source,redshift,chirp_mass_det
4,GW230627_015337,O4,6.02,0.07,6.4414
40,GW240622_004008,O4,12.50,0.28,16.0000
44,GW240920_073424,O4,15.60,0.27,19.8120
46,GW240922_142106,O4,8.12,0.24,10.0688
49,GW240925_005809,O4,6.85,0.08,7.3980
53,GW241102_124058,O4,8.12,0.08,8.7696
64,GW241231_054133,O4,8.08,0.18,9.5344
71,GW250119_190238,O4,9.38,0.07,10.0366



O3 events with detector-frame chirp mass > 100 Msun:


,name,run,chirp_mass_source,redshift,chirp_mass_det



O4 events with detector-frame chirp mass > 100 Msun:


,name,run,chirp_mass_source,redshift,chirp_mass_det
15,GW230922_040658,O4,52.0,0.96,101.92
21,GW231028_153006,O4,63.0,0.68,105.84
24,GW231123_135430,O4,101.0,0.40,141.40
